In [35]:
import numpy as np
import pyvista as pv
from scipy.spatial.distance import euclidean
pv.set_jupyter_backend('trame')

In [36]:
folder = 'D:/VMWare Shared Folder/230915_all_conditions_mito_decon/'
filename = 'S8_TRAK2_oe_014_mitosurface'

mesh = pv.read(folder + filename + '.vtk')

In [37]:
connected_components = mesh.connectivity()

# Calculate the cell centers for each connected component
centers = mesh.cell_centers

In [38]:
# Group cells by component label and calculate the sizes
component_labels = connected_components['RegionId']
unique_labels, label_counts = np.unique(component_labels, return_counts=True)
component_sizes = dict(zip(unique_labels, label_counts))

In [39]:
# Calculate the center of mass (centroid) for each connected component
centroids = {}
for label in unique_labels:
    mask = component_labels == label
    component_centers = mesh.extract_cells(mask)
    centroid = np.mean(component_centers.points, axis=0)
    centroids[label] = centroid

In [40]:
# Convert the centroids to a PyVista Points object
centroid_points = pv.PolyData(list(centroids.values()))

In [41]:
np_centroids = [np.array(centroid) for centroid in centroids.values()]
cell_centroid = np.mean(np_centroids, axis=0)

In [42]:
# Create a list to store component information (size and distance)
component_distance = []

# Calculate distances from the image centroid to individual component centroids
for centroid in np_centroids:

    distance = euclidean(cell_centroid, centroid)
    
    # Store component size and distance as a tuple
    component_distance.append(distance)

In [43]:
component_info = []
for key, value in component_sizes.items():
    element_from_list = component_distance.pop(0)
    component_info.append([key, value, element_from_list])

In [44]:
max_size = max(label_counts)

# Create an array of colors based on the component labels
colors = [(component_sizes[label]/max_size) for label in component_labels]

In [45]:
cell_centroid = pv.PolyData(cell_centroid)

# Create a plotter and display the PolyData with colors
plotter = pv.Plotter(notebook=True)
plotter.add_mesh(mesh, scalars=colors, cmap='turbo', smooth_shading=True)
#plotter.add_mesh(centroid_points, color='red', point_size=10)  # Visualize centroids as red points
#plotter.add_points(cell_centroid, render_points_as_spheres=True, color='blue', point_size=10)
plotter.camera.zoom(1.2)

#path = plotter.generate_orbital_path(n_points=60, shift=0.1, viewup=[0,1,0])
#plotter.open_gif('mitograph-decon/S1_Ctrl_secondRound000_mitosurface.gif')
#plotter.orbit_on_path(path, write_frames=True, viewup=[0,1,0])
#plotter.close()

#plotter.view_xy()
#plotter.screenshot('paper_plots/' + filename + '.png', window_size=[2000, 2000], return_img=False)

# Show the updated plot
plotter.show()

Widget(value="<iframe src='http://localhost:57371/index.html?ui=P_0x1c8b35542d0_4&reconnect=auto' style='width…